# HLLSet DSL — User Guide

This notebook is a hands-on guide to the HLLSet Algebra DSL using the
Rust API (`hllset-core`, `hllset-dsl`).

The same operations are available through the `hllset` CLI:
- `hllset -e '<lua>'` — execute Lua scripts
- `hllset --forth '<forth>'` — compile and run Forth DSL
- `hllset --repl` — interactive REPL

**Concepts:** tokenization, set algebra, BSS similarity, content-addressed
storage, temporal storage, materialization, Forth colon-definitions.

In [2]:
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-core" }
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-dsl" }
:dep hllset-forth = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-forth" }
:dep hllset-storage = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-storage" }

use hllset_core::*;
use hllset_dsl::{DslRuntime, LatticeElement, Tokenizer};
use hllset_storage::{MemoryStorage, Storage};

println!("hllset DSL loaded — M={M}, TOTAL_BITS={}", hllset_core::core::hllset::TOTAL_BITS);

hllset DSL loaded — M=1024, TOTAL_BITS=32768


---
## 1. Tokenization & HLLSet Creation

An HLLSet is created by *inscribing* tokens. Each token is hashed with
MurmurHash3, and the resulting bit position is set in a 32,768-bit bitmap.

**Key properties:**
- **Idempotent:** same token → same bits, no matter how many times
- **Content-addressed:** the SHA-1 hash of the bitmap IS the key (`h:<sha1>`)

In [3]:
let a = HLLSet::from_tokens(&["hello", "world", "lua"]);
let b = HLLSet::from_tokens(&["hello", "world", "lua"]);

println!("Key A: {}", a.content_key());
println!("Key B: {}", b.content_key());
println!("Same key? {}", a.content_key() == b.content_key());

// Idempotence: adding same token twice doesn't change the set
let mut h = HLLSet::new();
h.add_token(b"alice");
let after_one = h.popcount();
h.add_token(b"alice");
assert_eq!(h.popcount(), after_one, "idempotent insertion");
println!("Idempotent: popcount stays {} after re-adding", after_one);

Key A: h:e141b57a3d7b9b87165b907600337e6298633845
Key B: h:e141b57a3d7b9b87165b907600337e6298633845
Same key? true
Idempotent: popcount stays 1 after re-adding


### Tokenize text

The tokenizer supports word splitting, lowercase, n-grams, and boundary padding.

In [4]:
let tok = Tokenizer::new().lowercase();
let doc = tok.apply(b"The Cat Sat On The Mat");
println!("Key:  {}", doc.key());
println!("Card: {:.1}", doc.cardinality());
println!("Bits: {}", doc.popcount());

// n-gram tokenizer with boundary markers
let tok2 = Tokenizer::new().lowercase().pad(b"<S>", b"</S>").ngrams(1, 2);
let doc2 = tok2.apply(b"the cat sat");
let tokens = tok2.tokenize(b"the cat sat");
println!("n-grams: {} bits, tokens: {:?}", doc2.popcount(), tokens);

Key:  h:585f089a1aac49bfd8119bc8a7661b954ccb5df0
Card: 5.0
Bits: 5
n-grams: 9 bits, tokens: [[60, 83, 62], [116, 104, 101], [99, 97, 116], [115, 97, 116], [60, 47, 83, 62], [60, 83, 62, 0, 116, 104, 101], [116, 104, 101, 0, 99, 97, 116], [99, 97, 116, 0, 115, 97, 116], [115, 97, 116, 0, 60, 47, 83, 62]]


---
## 2. Set Algebra

HLLSets form a **bounded distributive lattice**:
- **Union** (join): `a.union(&b)` — bitwise OR
- **Intersection** (meet): `a.intersection(&b)` — bitwise AND
- **Difference**: `a.difference(&b)` — bitwise AND-NOT

All operations are associative, commutative, and idempotent.

In [5]:
let a = HLLSet::from_tokens(&["a", "b", "c"]);
let b = HLLSet::from_tokens(&["b", "c", "d"]);

let u = a.union(&b);
let i = a.intersection(&b);
let d = a.difference(&b);

println!("A: {} bits, B: {} bits", a.popcount(), b.popcount());
println!("Union:        {} bits", u.popcount());
println!("Intersection: {} bits", i.popcount());
println!("Difference:   {} bits", d.popcount());

// Algebraic laws
assert_eq!(a.union(&a).popcount(), a.popcount());
assert_eq!(a.union(&b).popcount(), b.union(&a).popcount());
assert!(u.popcount() >= a.popcount());
println!("All lattice laws hold");

A: 3 bits, B: 3 bits
Union:        4 bits
Intersection: 2 bits
Difference:   1 bits
All lattice laws hold


### Cardinality estimation (Horvitz-Thompson)

In [6]:
println!("1 token:  {:.1}", HLLSet::from_tokens(&["one"]).cardinality());
println!("3 tokens: {:.1}", HLLSet::from_tokens(&["a","b","c"]).cardinality());
println!("8 tokens: {:.1}", HLLSet::from_tokens(&["a","b","c","d","e","f","g","h"]).cardinality());

// Cardinality is monotonic
let mut h = HLLSet::new();
let mut prev = 0.0;
let mut ok = true;
for i in 0u32..200 {
    h.add_token(&i.to_le_bytes());
    let cur = h.cardinality();
    if cur < prev - 0.001 { ok = false; }
    prev = cur;
}
println!("Monotonic over 200 inserts: {}", ok);

1 token:  1.0
3 tokens: 3.0
8 tokens: 8.0
Monotonic over 200 inserts: true


---
## 3. BSS Similarity & Morphisms

**Bell State Similarity (BSS):**
- `bss_inclusion(B)`: τ = |A∩B| / |B| — how much of B is in A
- `bss_exclusion(B)`: ρ = |A\B| / |B| — how much of B is NOT in A
- `jaccard(B)`: |A∩B| / |A∪B|
- `morph_to(B, τ_min, ρ_max)`: does A structurally contain B?

In [7]:
let scene = LatticeElement::from_tokens(&["red", "car", "intersection"]);
let rule  = LatticeElement::from_tokens(&["slow", "down", "intersection"]);

let tau = scene.bss_inclusion(&rule);
let rho = scene.bss_exclusion(&rule);
let jac = scene.jaccard_similarity(&rule);

println!("BSS inclusion (tau): {:.3}", tau);
println!("BSS exclusion (rho): {:.3}", rho);
println!("Jaccard:             {:.3}", jac);

let m = scene.morph_to(&rule, 0.2, 0.9);
println!("Morphism holds: {}", m.morphism_holds);

BSS inclusion (tau): 0.333
BSS exclusion (rho): 0.667
Jaccard:             0.200
Morphism holds: true


---
## 4. Content-Addressed Storage

HLLSets are stored by their content key (`h:<sha1>`). Backends:
`MemoryStorage` (dev), `IpfrsNativeStorage` (sled), `RedisStorage` (prod).

In [8]:
let store = MemoryStorage::new();

let doc = LatticeElement::from_tokens(&["hello", "world", "from", "hllset"]);
let key = doc.key().to_string();
store.put(&key, &doc.hllset().to_bytes()).unwrap();
println!("Stored: {}", key);

assert!(store.has(&key).unwrap());
let loaded_bytes = store.get(&key).unwrap().unwrap();
let loaded = HLLSet::from_bytes(&loaded_bytes).unwrap();
assert_eq!(doc.hllset().content_key(), loaded.content_key());

let h_keys = store.list("h:").unwrap();
println!("Keys with 'h:' prefix: {}", h_keys.len());
println!("Roundtrip: store -> load -> verify");

Stored: h:622a9758a1e0fc9166bf22dc60057b1c39f79b2e
Keys with 'h:' prefix: 1
Roundtrip: store -> load -> verify


### Pin & Garbage Collect

In [9]:
let keep = HLLSet::from_tokens(&["important", "data"]);
let toss = HLLSet::from_tokens(&["temporary", "cache"]);

store.put(&keep.content_key(), &keep.to_bytes()).unwrap();
store.put(&toss.content_key(), &toss.to_bytes()).unwrap();
store.pin(&keep.content_key()).unwrap();

let removed = store.gc().unwrap();
println!("Removed by GC: {}", removed.len());
println!("Keep exists:  {}", store.has(&keep.content_key()).unwrap());
println!("Toss exists:  {}", store.has(&toss.content_key()).unwrap());

Removed by GC: 2
Keep exists:  true
Toss exists:  false


---
## 5. Temporal Storage

`put_tmp`/`get_tmp`/`cas_tmp` store mutable system state under named keys
like `system:tf`. NOT content-addressed — for TF vectors, head pointers,
and global state.

**CAS (compare-and-swap)** enables atomic updates without locks.

In [10]:
store.put_tmp("system:config", b"mode=production").unwrap();
let val = store.get_tmp("system:config").unwrap();
println!("Config: {:?}", val.as_ref().map(|v| String::from_utf8_lossy(v)));

// CAS: atomically swap old -> new
let ok = store.cas_tmp("system:config", b"mode=production", b"mode=staging").unwrap();
let new_val = store.get_tmp("system:config").unwrap();
println!("CAS ok: {}", ok);
println!("New: {:?}", new_val.map(|v| String::from_utf8_lossy(&v).to_string()));

Config: Some("mode=production")
CAS ok: true
New: Some("mode=staging")


### CAS rejection

In [11]:
store.put_tmp("system:counter", b"10").unwrap();
let result = store.cas_tmp("system:counter", b"wrong", b"20");
println!("CAS with wrong expected: {:?}", result);
let current = store.get_tmp("system:counter").unwrap();
println!("Unchanged: {:?}", current.map(|v| String::from_utf8_lossy(&v).to_string()));

CAS with wrong expected: Err(CasMismatch { expected: [119, 114, 111, 110, 103], actual: [49, 48] })
Unchanged: Some("10")


---
## 6. Materialization

Materialization recovers tokens from an HLLSet using a lookup table (LUT).
Since HLLSets are lossy (hash collisions), it reports a `confidence` score.

In [12]:
use hllset_dsl::materialize::{self, TokenLUT};

let tok = Tokenizer::new().lowercase();
let hllset = tok.apply(b"alpha beta gamma delta").into_hllset();
let tokens = tok.tokenize(b"alpha beta gamma delta");
let lut = TokenLUT::from_tokens(&tokens);

let result = materialize::materialize_inlut(&hllset, &lut);
println!("Strategy:   {}", result.strategy);
println!("Confidence: {:.3}", result.confidence);
println!("Tokens:     {:?}", result.flat_strings());

Strategy:   InLUT
Confidence: 1.000
Tokens:     ["delta", "beta", "gamma", "alpha"]


### De Bruijn reconstruction (n-gram data)

In [13]:
let tok = Tokenizer::new().lowercase().pad(b"<S>", b"</S>").ngrams(2, 2);
let hllset = tok.apply(b"the cat sat").into_hllset();
let tokens = tok.tokenize(b"the cat sat");
let lut = TokenLUT::from_tokens(&tokens);

let result = materialize::materialize_debruijn(&hllset, &lut, b"<S>", b"</S>");
println!("Strategy:   {}", result.strategy);
println!("Confidence: {:.3}", result.confidence);
println!("Tokens:     {:?}", result.flat_strings());

Strategy:   DeBruijnReconstruct
Confidence: 1.000
Tokens:     ["<S>", "the", "cat", "sat", "</S>"]


---
## 7. Forth DSL with Colon-Definitions

The Forth DSL compiles to Lua. It supports stack ops (DUP, SWAP, DROP),
HLLSet ops (INSCRIBE, UNION, INTERSECT, DIFF, BSS), and storage (STORE,
LOAD). **Colon-definitions** (`: NAME ... ;`) compile to Lua functions.

In [14]:
use hllset_forth::{parse, compile_to_lua};

// Colon-definition: compiles to a Lua function
let forth = r#": GREET "hello from forth" ;"#;
let ast = parse(forth).unwrap();
let lua = compile_to_lua(&ast);
println!("Forth: {}", forth);
println!("Lua:\n{}", lua);

Forth: : GREET "hello from forth" ;
Lua:
-- Generated by hllset-forth
function GREET()
v0 = "hello from forth"
  return v0
end
v0 = GREET()
return {v0}



In [15]:
// Compose HLLSet operations in Forth, run via Lua runtime
let forth = r#"
"red" "car" "intersection" 3 INSCRIBE
"slow" "down" "intersection" 3 INSCRIBE
INTERSECT CARD
"#;

let ast = parse(forth).unwrap();
let lua = compile_to_lua(&ast);
println!("Compiled Lua:\n{}", lua);

let rt = DslRuntime::new().unwrap();
let result: f64 = rt.eval(&lua).unwrap();
println!("Result (intersection cardinality): {:.1}", result);

Compiled Lua:
-- Generated by hllset-forth
v0 = "red"
v1 = "car"
v2 = "intersection"
v3 = 3
v4 = hllset.inscribe({ v0, v1, v2 })
v5 = "slow"
v6 = "down"
v7 = "intersection"
v8 = 3
v9 = hllset.inscribe({ v5, v6, v7 })
v10 = v4 * v9
v11 = #v10
return {v11}




thread '<unnamed>' (46244) panicked at src/lib.rs:186:33:
called `Result::unwrap()` on an `Err` value: FromLuaConversionError { from: "table", to: "f64", message: Some("expected number or string coercible to number") }
stack backtrace:
   0: __rustc::rust_begin_unwind
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library/std/src/panicking.rs:689:5
   1: core::panicking::panic_fmt
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library/core/src/panicking.rs:80:14
   2: core::result::unwrap_failed
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library/core/src/result.rs:1867:5
   3: run_user_code_14
   4: evcxr::runtime::Runtime::run_loop
   5: evcxr::runtime::runtime_hook
   6: evcxr_jupyter::main
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.


---
## 8. Lua Scripting with DslRuntime

The `DslRuntime` executes Lua scripts with the full `hllset` API.

In [16]:
let rt = DslRuntime::new().unwrap();

let card: f64 = rt.eval(r#"
    local a = hllset.inscribe({"hello", "world", "lua"})
    return #a
"#).unwrap();
println!("Cardinality: {:.1}", card);

let (u, i): (f64, f64) = rt.eval(r#"
    local a = hllset.inscribe({"a", "b", "c"})
    local b = hllset.inscribe({"b", "c", "d"})
    return #(a + b), #(a * b)
"#).unwrap();
println!("Union: {:.1}, Intersection: {:.1}", u, i);

Cardinality: 3.0
Union: 4.0, Intersection: 2.0


---
## 9. Summary: The IICA Pipeline

Every operation satisfies three properties simultaneously:

| Property | Meaning |
|----------|---------|
| **I**dempotency | f(x) = f(f(x)) — same input, same output |
| **I**mmutability | Once computed, never changes |
| **C**ontent-**A**ddressability | Output IS its address (`h:<sha1>`) |

These compose: if each step is IICA, the entire pipeline is IICA.
This is what makes nested spaces, distributed convergence, and
cross-domain bridges work without new theory.

In [17]:
let tokens = ["alice", "bob", "carol"];

// Idempotent: same input -> same key
let h1 = HLLSet::from_tokens(&tokens);
let h2 = HLLSet::from_tokens(&tokens);
assert_eq!(h1.content_key(), h2.content_key());

// Content-addressed: key IS the content hash
assert!(h1.content_key().starts_with("h:"));

// Union idempotent: A u A = A
assert_eq!(h1.union(&h1).popcount(), h1.popcount());

println!("IICA pipeline verified");
println!("Key: {}", h1.content_key());
println!("popcount(A u A) = popcount(A) = {}", h1.popcount());

IICA pipeline verified
Key: h:000a705032649d985683fcc9ae00732b5a7e7906
popcount(A u A) = popcount(A) = 3
